<a href="https://colab.research.google.com/github/mmbc560/GUIA2/blob/main/Evaluating_Supplier_Performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# AHP FOR GLOBAL SUPPLIER SELECTION
# ================================================================
#
# OBJETIVO
#
# Seleccionar el mejor proveedor utilizando
# Analytic Hierarchy Process - AHP.
#
# Vamos a comparar cuatro proveedores:
#
# 1. Alpha Global
# 2. Beta Industries
# 3. Gamma Supply
# 4. Delta Strategic
#
# utilizando seis criterios:
#
# 1. Cost
# 2. Lead Time
# 3. Delivery Reliability
# 4. Quality
# 5. Flexibility
# 6. Risk
#
# La idea principal es demostrar que:
#
# "The cheapest supplier is not necessarily the best supplier."
#
# ================================================================


# ================================================================
# 1. IMPORTAR LIBRERÍAS
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ================================================================
# 2. DEFINIR CRITERIOS Y ALTERNATIVAS
# ================================================================

criteria = [
    "Cost",
    "Lead Time",
    "Delivery Reliability",
    "Quality",
    "Flexibility",
    "Risk"
]


suppliers = [
    "Alpha Global",
    "Beta Industries",
    "Gamma Supply",
    "Delta Strategic"
]


# ================================================================
# 3. ESCALA DE SAATY
# ================================================================
#
# AHP utiliza comparaciones por pares.
#
# La escala tradicional de Saaty es:
#
# 1 = Igual importancia
# 3 = Importancia moderada
# 5 = Importancia fuerte
# 7 = Importancia muy fuerte
# 9 = Importancia extrema
#
# 2, 4, 6 y 8 son valores intermedios.
#
# Si A es 3 veces más importante que B,
# entonces B es 1/3 de importante respecto a A.
#
# Ejemplo:
#
#             Cost   Quality
# Cost          1       1/3
# Quality       3        1
#
# Significa que Quality es moderadamente
# más importante que Cost.
#
# ================================================================


# ================================================================
# 4. MATRIZ DE COMPARACIÓN DE LOS CRITERIOS
# ================================================================
#
# Aquí simulamos la opinión del equipo directivo.
#
# El equipo considera que:
#
# - Quality es uno de los criterios más importantes.
# - Delivery Reliability también es crítico.
# - Cost sigue siendo importante.
# - Risk y Flexibility tienen importancia intermedia.
# - Lead Time tiene menor peso relativo.
#
# IMPORTANTE:
#
# Esta matriz podría construirse en clase preguntando
# directamente a los estudiantes:
#
# "How much more important is Quality than Cost?"
#
# ================================================================


criteria_matrix = np.array([

# Cost   LT    Reliability Quality Flexibility Risk

[1,      3,    1/2,        1/3,    2,          2],       # Cost

[1/3,    1,    1/4,        1/5,    1/2,        1/2],     # Lead Time

[2,      4,    1,          1/2,    3,          3],       # Reliability

[3,      5,    2,          1,      4,          4],       # Quality

[1/2,    2,    1/3,        1/4,    1,          1],       # Flexibility

[1/2,    2,    1/3,        1/4,    1,          1]        # Risk

])


# ================================================================
# 5. FUNCIÓN PARA CALCULAR AHP
# ================================================================
#
# Esta función realizará automáticamente:
#
# 1. Cálculo del eigenvalor principal.
# 2. Cálculo del eigenvector.
# 3. Obtención de los pesos.
# 4. Consistency Index (CI).
# 5. Consistency Ratio (CR).
#
# ================================================================


def calculate_ahp(matrix):

    # ------------------------------------------------------------
    # PASO A.
    # Calculamos eigenvalores y eigenvectores
    # ------------------------------------------------------------

    eigenvalues, eigenvectors = np.linalg.eig(matrix)


    # ------------------------------------------------------------
    # PASO B.
    # Identificamos el eigenvalor principal.
    #
    # AHP utiliza el mayor eigenvalor de la matriz.
    # ------------------------------------------------------------

    max_index = np.argmax(eigenvalues.real)

    lambda_max = eigenvalues[max_index].real


    # ------------------------------------------------------------
    # PASO C.
    # Extraemos el eigenvector asociado.
    # ------------------------------------------------------------

    principal_vector = np.abs(
        eigenvectors[:, max_index].real
    )


    # ------------------------------------------------------------
    # PASO D.
    # Normalizamos el vector.
    #
    # Queremos que todos los pesos sumen 1.
    # ------------------------------------------------------------

    weights = (
        principal_vector /
        principal_vector.sum()
    )


    # ------------------------------------------------------------
    # PASO E.
    # CALCULAR CONSISTENCY INDEX - CI
    # ------------------------------------------------------------
    #
    # Fórmula:
    #
    # CI = (lambda_max - n) / (n - 1)
    #
    # ------------------------------------------------------------

    n = matrix.shape[0]

    CI = (
        (lambda_max - n) /
        (n - 1)
    )


    # ------------------------------------------------------------
    # PASO F.
    # RANDOM INDEX - RI
    # ------------------------------------------------------------
    #
    # Valores propuestos por Saaty.
    # ------------------------------------------------------------

    RI_table = {

        1: 0.00,
        2: 0.00,
        3: 0.58,
        4: 0.90,
        5: 1.12,
        6: 1.24,
        7: 1.32,
        8: 1.41,
        9: 1.45,
        10: 1.49
    }


    RI = RI_table[n]


    # ------------------------------------------------------------
    # PASO G.
    # CONSISTENCY RATIO - CR
    # ------------------------------------------------------------
    #
    # Fórmula:
    #
    # CR = CI / RI
    #
    # Regla general:
    #
    # CR < 0.10  --> comparación aceptable
    # CR >= 0.10 --> revisar los juicios
    #
    # ------------------------------------------------------------

    if RI == 0:

        CR = 0

    else:

        CR = CI / RI


    return weights, lambda_max, CI, CR


# ================================================================
# 6. CALCULAR PESOS DE LOS CRITERIOS
# ================================================================


criteria_weights, lambda_max, CI, CR = calculate_ahp(
    criteria_matrix
)


# Creamos una tabla sencilla.

criteria_results = pd.DataFrame({

    "Criterion": criteria,

    "Weight": criteria_weights,

    "Weight (%)": criteria_weights * 100

})


criteria_results = criteria_results.sort_values(
    "Weight",
    ascending=False
)


print("\n" + "=" * 70)

print("STEP 1 - CRITERIA WEIGHTS")

print("=" * 70)


print(

    criteria_results
    .round(3)
    .to_string(index=False)

)


print("\nConsistency Ratio:")

print(
    f"CR = {CR:.4f}"
)


if CR < 0.10:

    print(
        "Result: ACCEPTABLE CONSISTENCY"
    )

else:

    print(
        "Result: REVIEW THE COMPARISONS"
    )


# ================================================================
# 7. COMPARAR PROVEEDORES PARA CADA CRITERIO
# ================================================================
#
# Ahora necesitamos preguntarnos:
#
# "Which supplier performs better under each criterion?"
#
# Para cada criterio se construye una matriz AHP.
#
# ================================================================


# ================================================================
# COST
# ================================================================
#
# Alpha tiene el mejor costo.
# Gamma tiene el costo menos atractivo.
#
# ================================================================

cost_matrix = np.array([

    [1,   3,   5,   2],

    [1/3, 1,   2,   1/2],

    [1/5, 1/2, 1,   1/3],

    [1/2, 2,   3,   1]

])


# ================================================================
# LEAD TIME
# ================================================================
#
# Beta tiene muy buen Lead Time.
# Gamma presenta el peor tiempo de entrega.
#
# ================================================================

leadtime_matrix = np.array([

    [1,   1/3, 3,   1/2],

    [3,   1,   5,   2],

    [1/3, 1/5, 1,   1/4],

    [2,   1/2, 4,   1]

])


# ================================================================
# DELIVERY RELIABILITY
# ================================================================
#
# Delta es el proveedor más confiable.
#
# ================================================================

reliability_matrix = np.array([

    [1,   1/3, 1/2, 1/4],

    [3,   1,   2,   1/2],

    [2,   1/2, 1,   1/3],

    [4,   2,   3,   1]

])


# ================================================================
# QUALITY
# ================================================================
#
# Delta muestra el mejor desempeño en calidad.
#
# ================================================================

quality_matrix = np.array([

    [1,   1/2, 2,   1/3],

    [2,   1,   3,   1/2],

    [1/2, 1/3, 1,   1/4],

    [3,   2,   4,   1]

])


# ================================================================
# FLEXIBILITY
# ================================================================
#
# Delta tiene mayor capacidad de reacción
# frente a cambios en volumen o condiciones.
#
# ================================================================

flexibility_matrix = np.array([

    [1,   2,   1/2, 1/3],

    [1/2, 1,   1/3, 1/4],

    [2,   3,   1,   1/2],

    [3,   4,   2,   1]

])


# ================================================================
# RISK
# ================================================================
#
# IMPORTANTE:
#
# En este criterio, una mayor prioridad significa
# mejor desempeño frente al riesgo,
# es decir, MENOR riesgo.
#
# Delta aparece como el proveedor más seguro.
#
# ================================================================

risk_matrix = np.array([

    [1,   1/2, 2,   1/3],

    [2,   1,   3,   1/2],

    [1/2, 1/3, 1,   1/4],

    [3,   2,   4,   1]

])


# ================================================================
# 8. GUARDAR TODAS LAS MATRICES
# ================================================================


alternative_matrices = {

    "Cost": cost_matrix,

    "Lead Time": leadtime_matrix,

    "Delivery Reliability": reliability_matrix,

    "Quality": quality_matrix,

    "Flexibility": flexibility_matrix,

    "Risk": risk_matrix

}


# ================================================================
# 9. CALCULAR LOS PESOS DE LOS PROVEEDORES
# ================================================================


local_priorities = {}


print("\n" + "=" * 70)

print("STEP 2 - SUPPLIER PRIORITIES BY CRITERION")

print("=" * 70)


for criterion, matrix in alternative_matrices.items():

    weights, lam, ci, cr = calculate_ahp(
        matrix
    )


    local_priorities[
        criterion
    ] = weights


    print(
        f"\nCriterion: {criterion}"
    )


    temp = pd.DataFrame({

        "Supplier": suppliers,

        "Priority": weights

    })


    print(

        temp
        .sort_values(
            "Priority",
            ascending=False
        )
        .round(3)
        .to_string(index=False)

    )


    print(
        f"CR = {cr:.4f}"
    )


# ================================================================
# 10. CREAR MATRIZ DE PRIORIDADES
# ================================================================
#
# Cada columna representa un criterio.
# Cada fila representa un proveedor.
#
# ================================================================


priority_matrix = pd.DataFrame(

    local_priorities,

    index=suppliers

)


print("\n" + "=" * 70)

print("LOCAL PRIORITY MATRIX")

print("=" * 70)


print(

    priority_matrix
    .round(3)

)


# ================================================================
# 11. CALCULAR PUNTAJE FINAL
# ================================================================
#
# Ahora combinamos:
#
# prioridad del proveedor
#
# ×
#
# peso del criterio
#
# ================================================================


# Primero colocamos los pesos de criterios
# en el mismo orden de las columnas.

criteria_weights_series = pd.Series(

    criteria_weights,

    index=criteria

)


# Multiplicamos cada desempeño local
# por el peso correspondiente del criterio.

weighted_matrix = (

    priority_matrix *
    criteria_weights_series

)


# Sumamos horizontalmente.

final_scores = weighted_matrix.sum(
    axis=1
)


# Creamos tabla final.

final_ranking = pd.DataFrame({

    "Supplier": suppliers,

    "Final Score": final_scores.values,

    "Final Score (%)":
        final_scores.values * 100

})


final_ranking = final_ranking.sort_values(

    "Final Score",

    ascending=False

)


print("\n" + "=" * 70)

print("STEP 3 - FINAL SUPPLIER RANKING")

print("=" * 70)


print(

    final_ranking
    .round(3)
    .to_string(index=False)

)


# ================================================================
# 12. IDENTIFICAR EL GANADOR
# ================================================================


best_supplier = final_ranking.iloc[0][
    "Supplier"
]


best_score = final_ranking.iloc[0][
    "Final Score (%)"
]


print("\n" + "=" * 70)

print("AHP DECISION")

print("=" * 70)


print(

    f"""
Recommended Supplier:

{best_supplier}

Final AHP Score:

{best_score:.2f}%

This supplier provides the best overall balance
across cost, lead time, reliability, quality,
flexibility and risk.
"""

)


# ================================================================
# 13. GRÁFICA DE BARRAS
# FINAL SUPPLIER RANKING
# ================================================================


ranking_plot = final_ranking.sort_values(
    "Final Score"
)


colors = [
    "#E74C3C",
    "#F39C12",
    "#3498DB",
    "#2ECC71"
]


plt.figure(
    figsize=(11,6)
)


bars = plt.barh(

    ranking_plot["Supplier"],

    ranking_plot["Final Score (%)"],

    color=colors

)


plt.title(

    "AHP Supplier Selection - Final Ranking",

    fontsize=17,

    fontweight="bold"

)


plt.xlabel(
    "Final AHP Score (%)"
)


# Agregamos el porcentaje en cada barra.

for bar in bars:

    width = bar.get_width()

    plt.text(

        width + 0.5,

        bar.get_y() +
        bar.get_height()/2,

        f"{width:.1f}%",

        va="center",

        fontweight="bold"

    )


plt.grid(
    axis="x",
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# 14. RADAR CHART 1
# IMPORTANCE OF CRITERIA
# ================================================================
#
# Esta gráfica muestra visualmente
# cuáles criterios tienen mayor importancia.
#
# ================================================================


# Número de criterios.

N = len(criteria)


# Creamos los ángulos del radar.

angles = np.linspace(
    0,
    2 * np.pi,
    N,
    endpoint=False
).tolist()


# Cerramos el polígono.

angles += angles[:1]


# Valores de los pesos.

values = list(
    criteria_weights * 100
)


values += values[:1]


fig = plt.figure(
    figsize=(9,9)
)


ax = plt.subplot(
    111,
    polar=True
)


ax.plot(

    angles,

    values,

    linewidth=2.5,

    color="#2980B9"

)


ax.fill(

    angles,

    values,

    color="#3498DB",

    alpha=0.25

)


ax.set_xticks(
    angles[:-1]
)


ax.set_xticklabels(
    criteria,
    fontsize=10
)


ax.set_title(

    "AHP Criteria Importance",

    fontsize=17,

    fontweight="bold",

    pad=25

)


plt.show()


# ================================================================
# 15. RADAR CHART 2
# SUPPLIER PERFORMANCE PROFILE
# ================================================================
#
# Ahora comparamos los cuatro proveedores
# simultáneamente.
#
# Cada eje representa un criterio.
#
# Cuanto más hacia afuera esté un proveedor,
# mejor desempeño relativo presenta.
#
# ================================================================


supplier_colors = {

    "Alpha Global": "#E74C3C",

    "Beta Industries": "#F39C12",

    "Gamma Supply": "#8E44AD",

    "Delta Strategic": "#2ECC71"
}


fig = plt.figure(
    figsize=(10,10)
)


ax = plt.subplot(
    111,
    polar=True
)


for supplier in suppliers:

    # Obtenemos los valores locales
    # del proveedor para cada criterio.

    values = (
        priority_matrix
        .loc[supplier]
        .values
        * 100
    ).tolist()


    # Cerramos el radar.

    values += values[:1]


    ax.plot(

        angles,

        values,

        linewidth=2.2,

        label=supplier,

        color=supplier_colors[
            supplier
        ]

    )


    ax.fill(

        angles,

        values,

        alpha=0.08,

        color=supplier_colors[
            supplier
        ]

    )


ax.set_xticks(
    angles[:-1]
)


ax.set_xticklabels(
    criteria,
    fontsize=10
)


ax.set_title(

    "Supplier Performance Profile",

    fontsize=17,

    fontweight="bold",

    pad=25

)


ax.legend(

    loc="upper right",

    bbox_to_anchor=(1.35, 1.15)

)


plt.show()


# ================================================================
# 16. PREGUNTAS PARA DISCUSIÓN GERENCIAL
# ================================================================


print("\n" + "=" * 70)

print("MANAGEMENT DISCUSSION")

print("=" * 70)


print("""

1. Is the lowest-cost supplier the best supplier?

2. Which criterion has the greatest influence
   on the final decision?

3. Which supplier presents the best balance
   between cost and operational performance?

4. Would the ranking change if Cost became
   the most important criterion?

5. What would happen if Risk received twice
   its current weight?

6. Which supplier would you select
   if your priority were resilience?

7. Would you select one supplier
   or design a dual-sourcing strategy?

""")